# Pixel Relational Motif — E0.1b diagnostic

Post-hoc diagnostic for Issue #76. This does **not** overwrite the registered E0.1-v533 negative verdict. It reuses the exact v533 dictionary, measures recurrence/non-dominance on frozen Train_anchor, and uses PublicTest **pixels only** for out-of-sample recurrence. Emotion labels are ignored; PrivateTest is forbidden.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-motif-e01b'
SOURCE_SHA = '3858b3e17b9ea6cc591381e3096be53c27581c02'
EXPECTED_DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
FER_ROOT = Path('/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split')
TRAIN_CSV = FER_ROOT / 'train.csv'
PUBLIC_CSV = FER_ROOT / 'val.csv'
OUTPUT_DIR = Path('/kaggle/working/outputs/pixel_relational_motif_e01b')
PACKAGE_RELATIVE = Path('research/pixel_relational_motif_e0')
RUN_TESTS = True
RUN_E01B = True


In [ ]:
import hashlib, os, shutil, subprocess, sys
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'FER2013_Graph_E01B'
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_SHA: raise RuntimeError(f'source lock mismatch: {head} != {SOURCE_SHA}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
subprocess.run(['git','-C',str(PROJECT),'diff','--cached','--quiet'], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents: raise RuntimeError(f'import isolation violation: {imported}')
print('Source lock PASS:', head)


In [ ]:
if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    r = subprocess.run([sys.executable,'-m','pytest',str(PACKAGE/'tests'),'-q'], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if r.returncode != 0: raise RuntimeError(f'pytest failed: {r.returncode}')


In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

if not TRAIN_CSV.is_file(): raise FileNotFoundError(TRAIN_CSV)
if not PUBLIC_CSV.is_file(): raise FileNotFoundError(PUBLIC_CSV)
candidates = sorted(Path('/kaggle/input').rglob('e01_dictionary.npz'))
matches = [p for p in candidates if sha256(p) == EXPECTED_DICTIONARY_SHA256]
print('dictionary candidates:', [str(p) for p in candidates])
if len(matches) != 1:
    raise RuntimeError(f'need exactly one attached v533 e01_dictionary.npz with SHA {EXPECTED_DICTIONARY_SHA256}; matches={matches}')
DICTIONARY_NPZ = matches[0]
print('v533 dictionary lock PASS:', DICTIONARY_NPZ)
print('PublicTest pixels will be read label-blind from val.csv. No PrivateTest path is configured.')


In [ ]:
import contextlib, threading, time
@contextlib.contextmanager
def heartbeat(label, interval_seconds=180):
    stop=threading.Event(); started=time.time()
    def worker():
        while not stop.wait(interval_seconds): print(f'[heartbeat] {label}: {(time.time()-started)/60:.1f} min', flush=True)
    t=threading.Thread(target=worker,daemon=True); t.start()
    try: yield
    finally:
        stop.set(); t.join(timeout=1); print(f'[heartbeat] {label}: complete {(time.time()-started)/60:.1f} min', flush=True)


In [ ]:
import json
from pixel_relational_motif_e0.e01b_diagnostic import run_e01b
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_E01B:
    with heartbeat('E0.1b recurrence/non-dominance'):
        summary = run_e01b(TRAIN_CSV, PUBLIC_CSV, DICTIONARY_NPZ, OUTPUT_DIR)
    print(json.dumps({
        'status':summary['status'],
        'registered_verdict_overwritten':summary['registered_verdict_overwritten'],
        'candidate_component_count':summary['candidate_component_count'],
        'anchor_conditional_nondominance_pass_count':summary['anchor_conditional_nondominance_pass_count'],
        'public_conditional_nondominance_pass_count':summary['public_conditional_nondominance_pass_count'],
        'cross_set':summary['cross_set'],
        'public_test_labels_used':summary['public_test_labels_used'],
        'private_test_read':summary['private_test_read'],
    }, indent=2))


In [ ]:
artifacts = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())
print('Artifacts:', artifacts)
required={'e01b_summary.json','e01b_counts.npz'}
missing=sorted(required-set(artifacts))
if missing: raise RuntimeError(f'missing E0.1b artifacts: {missing}')
print('E0.1b diagnostic complete. Registered E0.1-v533 verdict remains unchanged.')
